# Bayesian optimization of the reaction yield for a real-world dataset

In [ ]:
import baybe

import os
import pandas as pd
import seaborn as sns
from matplotlib import pyplot as plt

os.environ['BAYBE_CACHE_DIR']='' # turn descriptor caching off
from baybe.utils.random import set_random_seed
set_random_seed(1337)

# Load in the dataset 
The problem we study here, corresponds to the chemical reaction yield optimization from [Shields, B.J., Stevens et al. Nature 590, 89–96 (2021)](https://doi.org/10.1038/s41586-021-03213-y).

![reaction](https://raw.githubusercontent.com/emdgroup/baybe-ac24-workshop/main/files/reaction.png)

The parameters screened are:
- Solvent
- Base
- Ligand
- Concentration (of the solvent)
- Temperature

There is one target that should be maximized:
- yield

We will load in the high-throughput dataset as a lookup table, so that we do not need to run the experiments ourselves. Instead, we can look up the experimental yield for any combination of reaction parameters.

In [ ]:
lookup = pd.read_excel('shields_dataset.xlsx', index_col=0)
F_BEST = lookup['yield'].max()
lookup.head()

### SMILES
Note that the structure of the molecules is represented by SMILES strings.

In [ ]:
# TODO: visualize the first ligand and solvent SMILES encountered in the dataset.
from rdkit import Chem
from rdkit.Chem import Draw

ligand = Chem.MolFromSmiles('P(C1=CC=CC=C1)(C2=CC=CC=C2)C3=CC=CC=C3') #PPh3
Draw.MolToImage(ligand)

In [ ]:
solvent = Chem.MolFromSmiles('CC(N(C)C)=O')
Draw.MolToImage(solvent)

### Data Exploration
It is important to realize that optimizing the yield essentially corresponds to solving a needle-in-a-haystack problem: the data distribution is very much skewed towards low yields in the lookup dataset.

In [ ]:
sns.histplot(data = lookup, x = 'yield', edgecolor = 'black');
plt.title('Reaction Yield Histogram');

### Setting up the search space

Just as in the introductory example, we first need to setup the search space. Instead of manually defining this here, we will extract it from the lookup dataset.

In [ ]:
solvent_data = dict(sorted(set(zip(lookup.Solvent, lookup.Solvent_SMILES))))
solvent_data

In [ ]:
base_data = dict(sorted(set(zip(lookup.Base, lookup.Base_SMILES))))
base_data

In [ ]:
ligand_data = dict(sorted(set(zip( lookup.Ligand, lookup.Ligand_SMILES))))
ligand_data

In [ ]:
temperature_values = set(lookup.Temp_C)
temperature_values

In [ ]:
concentration_values = set(lookup.Concentration)
concentration_values

## Substance featurizations: One hot encoding

To demonstrate how featurization of substances impact the speed at which the model is able to locate good reaction conditions to maximize the yield, we will start by considering a One Hot Encoding (OHE) featurization, i.e., every individual solvent, base and ligand is assigned a separate dimension in the search space (with only two possible values: 0 if the substance is not used in this experiment, and 1 if it is). Temperature and concentration are encoded as numerical discrete parameters.

In [ ]:
from baybe.parameters import (SubstanceParameter, 
                              CustomDiscreteParameter, 
                              CategoricalParameter, 
                              NumericalDiscreteParameter)

p_solvent_ohe = CategoricalParameter(name = "Solvent",
                                    values = solvent_data.keys(),
                                    encoding = 'OHE')
p_base_ohe = CategoricalParameter(name = "Base",
                                    values = base_data.keys(),
                                    encoding = 'OHE')
p_ligand_ohe = CategoricalParameter(name = "Ligand",
                                    values = ligand_data.keys(),
                                    encoding = 'OHE') # INT or OHE


with pd.option_context('display.max_columns', 5):
    display(p_ligand_ohe.comp_df.head())

In [ ]:
# Discrete numerical parameters
p_temp = NumericalDiscreteParameter(name = "Temp_C", values = temperature_values)
p_concentration = NumericalDiscreteParameter(name = "Concentration", values = concentration_values)

ohe_parameters = [
    p_solvent_ohe,
    p_base_ohe, 
    p_ligand_ohe,
    p_concentration, 
    p_temp
]

In [ ]:
from baybe.searchspace import SearchSpace

searchspace_ohe = SearchSpace.from_product(parameters=ohe_parameters)

We also have to set up the target and the objective

In [ ]:
from baybe.objectives import SingleTargetObjective
from baybe.targets import NumericalTarget

# TODO
yield_target = NumericalTarget(name = 'yield', mode = 'MAX')
objective = SingleTargetObjective(target = yield_target)

Let's now set up a campaign; we will not specify the recommender this time, but simply adopt the default settings from BayBE (you can take a look through the scource code if you want to know the details).

In [ ]:
from baybe.campaign import Campaign

ohe_campaign = Campaign(objective = objective,
                    searchspace = searchspace_ohe)

Instead of manually setting up the campaign and looking up the result for each recommendation, we will make use of some handy functionality in BayBE to simulate the campaign in full for us.

In [ ]:
from baybe.simulation import simulate_scenarios

MC_RUNS = 10 # this will take fairly long on typical free-of-charge cloud compute
NUMBER_ITERATIONS = 16
BATCH_SIZE = 2

# Run the utility for backtesting
results_ohe = simulate_scenarios(
    {'OHE': ohe_campaign},
    lookup, # the initial dataframe with the experimental results is our lookup
    batch_size = BATCH_SIZE, # how many experiments to perform in one batch
    n_doe_iterations = NUMBER_ITERATIONS, # how many batches to select successively
    n_mc_iterations = MC_RUNS, # number of Monte Carlo iterations -> this corresponds to the number of times we take a new random starting point 
                                # and run the full campaign from scratch (to get a statistically) meaningful sense about the merit of the featurization approach
)
results_ohe.head()

To visualize the campaign, we will define an auxiliary function.

In [ ]:
PLOTARGS = {
    'linestyle': 'solid',
    'marker': 'o',
    'markersize': 6, 
    'markeredgecolor': 'none'
}
FIGSIZE = (11,6)

def plot_campaign(results):
    results.rename(columns = {"Scenario": "Ligand Encoding"}, inplace = True)
    sns.lineplot(data = results, 
             x = "Num_Experiments", 
             y = "yield_CumBest", 
             hue = "Ligand Encoding",
             **PLOTARGS)

    plt.axhline(y = F_BEST, color = 'red', linestyle = '--', label = 'Best Possible')
    plt.gcf().set_size_inches(FIGSIZE)
    plt.gca().set_ylim(plt.gca().get_ylim()[0], F_BEST+5)
    plt.title('Influence of the Ligand Substance Encoding'); 

In [ ]:
plot_campaign(results_ohe)

In the plot above, you can see visually see how throughout the campaign, we gradually manage to increase the yield (the blue line is the average best yield recorded so far after N experiments across the 10 Monte Carlo campaign simulations; the shaded area provides the standard deviation).

Let's now compare this Bayesian Optimization campaign to a random sampling, i.e., we randomly select points in the search space to evaluate, instead of using an acquisition function.

## Substance featurizations: Mordred encoding

In [ ]:
from baybe.recommenders import RandomRecommender

random_campaign = Campaign(objective = objective,
                       searchspace = searchspace_ohe,
                       recommender = RandomRecommender())

In [ ]:
# Run the utility for backtesting
results_random = simulate_scenarios(
    {'random': random_campaign},
    lookup, # the initial dataframe with the experimental results is our lookup
    batch_size = BATCH_SIZE,
    n_doe_iterations = NUMBER_ITERATIONS,
    n_mc_iterations = MC_RUNS,
)
results_random.head()

In [ ]:
plot_campaign(results_random)

This hardly seems any better than when we used the one-hot encoding representation of the compounds!

Let us now consider instead of a one-hot encoding a more chemically meaningful encoding that enables the establishment of "similarity" between substances. We will specifically focus on the MORDRED representation, but others are also available in BayBE.

In [ ]:
# TODO: Now do the same for random and for Mordred featurization
p_solvent_mordred = SubstanceParameter(name = "Solvent",
                                    data = solvent_data,
                                    encoding = 'MORDRED')
p_base_mordred = SubstanceParameter(name = "Base",
                                    data = base_data,
                                    encoding = 'MORDRED')
p_ligand_mordred = SubstanceParameter(name = "Ligand",
                                    data = ligand_data,
                                    encoding = 'MORDRED')


mordred_parameters = [
    p_solvent_mordred,
    p_base_mordred, 
    p_ligand_mordred,
    p_concentration, 
    p_temp
]

searchspace_mordred = SearchSpace.from_product(parameters=mordred_parameters)

with pd.option_context('display.max_columns', 5):
    display(p_ligand_mordred.comp_df.head())

mordred_campaign = Campaign(objective = objective,
                    searchspace = searchspace_mordred)

# Run the utility for backtesting
results_mordred = simulate_scenarios(
    {'mordred': mordred_campaign},
    lookup, # the initial dataframe with the experimental results is our lookup
    batch_size = BATCH_SIZE, # how many experiments to perform in one batch
    n_doe_iterations = NUMBER_ITERATIONS, # how many batches to select successively
    n_mc_iterations = MC_RUNS, # number of Monte Carlo iterations -> this corresponds to the number of times we take a new random starting point 
                                # and run the full campaign from scratch (to get a statistically) meaningful sense about the merit of the featurization approach
)
results_mordred.head()

In [ ]:
plot_campaign(results_mordred)

This looks a lot better already! Let's now plot all 3 types of campaigns together in one plot.

In [ ]:
results_combined = pd.concat([results_ohe, results_random, results_mordred], ignore_index=True)
plot_campaign(results_combined)

Clearly, the choice of encoding matter: the Mordred campaign found the optimal yield rapidly, and clearly outperforms the other approaches tried out.